# 13.1. 대칭 조작과 점군

In [1]:
import numpy as np


def C(n, axis="z"):
    """z, y, x 축에 대한 360/n 도 회전"""
    th = 2 * np.pi / n
    c, s = np.cos(th), np.sin(th)
    if axis == "z":
        return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1.0]])
    if axis == "y":
        return np.array([[c, 0, s], [0, 1.0, 0], [-s, 0, c]])
    return np.array([[1.0, 0, 0], [0, c, -s], [0, s, c]])


def sigma(plane):
    """xy, xz, yz 평면에 대한 반사"""
    m = np.eye(3)
    m[{"yz": 0, "xz": 1, "xy": 2}[plane]] *= -1
    return m


def S(n, axis="z"):
    """회전반사: 회전한 다음 그 축에 수직인 면에서 반사"""
    perp = {"z": "xy", "y": "xz", "x": "yz"}[axis]
    return sigma(perp) @ C(n, axis)


E = np.eye(3)
i = -np.eye(3)

In [2]:
r, theta = 0.9572, np.radians(104.52)
water = np.array([
    [0, 0, 0],
    [0,  r * np.sin(theta / 2), -r * np.cos(theta / 2)],
    [0, -r * np.sin(theta / 2), -r * np.cos(theta / 2)],
])
labels = ["O", "H", "H"]


def is_symmetry(op, coords, labels, tol=1e-6):
    moved = coords @ op.T
    used = set()
    for p, l in zip(moved, labels):
        for j, (q, m) in enumerate(zip(coords, labels)):
            if j not in used and l == m and np.allclose(p, q, atol=tol):
                used.add(j)
                break
        else:
            return False
    return True


tests = {
    "E": E,
    "C2(z)": C(2),
    "C3(z)": C(3),
    "σ(xz)": sigma("xz"),
    "σ(yz)": sigma("yz"),
    "σ(xy)": sigma("xy"),
    "i": i,
}

for name, op in tests.items():
    print(f"{name:>5}: {'O' if is_symmetry(op, water, labels) else 'X'}")

    E: O
C2(z): O
C3(z): X
σ(xz): O
σ(yz): O
σ(xy): X
    i: X


In [3]:
methane = np.array(
    [
        [0.0, 0.0, 0.0],  # C
        [1.0, 1.0, 1.0],  # H
        [1.0, -1.0, -1.0],  # H
        [-1.0, 1.0, -1.0],  # H
        [-1.0, -1.0, 1.0],  # H
    ],
)
mlabels = ["C", "H", "H", "H", "H"]

mtests = {
    "E": E,
    "C2(z)": C(2),
    "C4(z)": C(4),
    "S4(z)": S(4),
    "i": i,
}

for name, op in mtests.items():
    print(f"{name:>5}: {'O' if is_symmetry(op, methane, mlabels) else 'X'}")

    E: O
C2(z): O
C4(z): X
S4(z): O
    i: X


In [4]:
ops = {
    "E": E,
    "C2": C(2),
    "σ_v": sigma("xz"),
    "σ_v'": sigma("yz"),
}


def which(M, table):
    for name, op in table.items():
        if np.allclose(M, op):
            return name
    return "??"


w = 7
print(" " * w + "".join(f"{k:>{w}}" for k in ops))
for a, A in ops.items():
    print(f"{a:>{w}}" + "".join(f"{which(A @ B, ops):>{w}}" for B in ops.values()))

             E     C2    σ_v   σ_v'
      E      E     C2    σ_v   σ_v'
     C2     C2      E   σ_v'    σ_v
    σ_v    σ_v   σ_v'      E     C2
   σ_v'   σ_v'    σ_v     C2      E
